# WC26 — Prediction Model Deep Dive

New showcase notebook. Covers all four models end-to-end:

| § | Model | What it covers |
|---|-------|----------------|
| 1 | All | Build the training match matrix from WC26 + history |
| 2 | Elo | Train + plot rating distribution and matchup predictions |
| 3 | Dixon-Coles | MLE fit, parameter interpretation, scoreline matrix |
| 4 | XGBoost | Feature engineering, training, SHAP importances, LOOCV |
| 5 | Poisson | Per-player ratings, team aggregation, scoreline matrix |
| 6 | All | Side-by-side model comparison on remaining fixtures |
| 7 | MC | Winner probability simulation — how each model contributes |
| 8 | Calibration | (Post-tournament) Predicted probability vs actual outcome |

Run `make etl-train` first to generate `dc_params.json`, `elo_ratings.json`,
`xgb_model.ubj`, and `xgb_features.json` in `data/`.


In [ ]:
import sys, json, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)

REPO = Path("..").resolve()
DATA = REPO / "data"
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from atwc26_core import config
config.DATA_DIR.__class__  # ensure import works

print("Repo root:", REPO)
print("Data dir :", DATA)
print("Artifacts present:")
for name in ["elo_ratings.json","dc_params.json","xgb_model.ubj","xgb_features.json",
             "winner_probabilities.json","player_profiles.parquet"]:
    mark = "✓" if (DATA/name).exists() else "✗ (run make etl-train)"
    print(f"  {mark}  {name}")


## §1 Training data: match matrix

In [ ]:
from etl.train.features import build_match_matrix, add_rolling_form

match_df = build_match_matrix(DATA / "all_players_stats.parquet",
                               DATA / "historical_form.parquet")
print(f"Match matrix: {match_df.shape}")
print(f"Date range  : {match_df.match_date.min().date()} → {match_df.match_date.max().date()}")
print(f"Outcome dist: {match_df['outcome'].map({2:'home win',1:'draw',0:'away win'}).value_counts().to_dict()}")
display(match_df.head(5))


In [ ]:
# Outcome distribution pie
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

match_df["outcome"].map({2:"Home win",1:"Draw",0:"Away win"}).value_counts().plot(
    kind="pie", autopct="%1.0f%%", ax=axes[0],
    colors=["#1D9E75","#EF9F27","#E24B4A"], startangle=140)
axes[0].set_title("Match outcomes (426 games)")
axes[0].set_ylabel("")

# Goals per game
goals = (match_df["h_goals"] + match_df["a_goals"])
axes[1].hist(goals, bins=range(0, 11), color="#3B8BD4", edgecolor="white", rwidth=0.85)
axes[1].set_xlabel("Total goals")
axes[1].set_ylabel("Matches")
axes[1].set_title(f"Goals per game (mean={goals.mean():.2f})")
plt.tight_layout()
plt.savefig("match_matrix_summary.png", dpi=130, bbox_inches="tight")
plt.show()


## §2 Elo model

In [ ]:
from etl.train.elo import train_elo, save_elo, ELO_START, K

# Train from scratch (same as make etl-train)
elo_ratings = train_elo(match_df)
print(f"Trained Elo for {len(elo_ratings)} teams")

elo_df = (pd.DataFrame({"team": list(elo_ratings.keys()),
                          "elo":  list(elo_ratings.values())})
            .sort_values("elo", ascending=False)
            .reset_index(drop=True))
elo_df["rank"] = range(1, len(elo_df)+1)

fig, ax = plt.subplots(figsize=(10, 6))
top = elo_df.head(20)
colors = ["#1D9E75" if e > ELO_START+100 else "#3B8BD4" if e > ELO_START
          else "#888780" for e in top["elo"]]
ax.barh(top["team"][::-1], top["elo"][::-1], color=colors[::-1], edgecolor="none")
ax.axvline(ELO_START, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Elo Rating")
ax.set_title("Elo Rankings — Top 20")
plt.tight_layout()
plt.savefig("elo_top20.png", dpi=130, bbox_inches="tight")
plt.show()
print(elo_df.head(16).to_string(index=False))


In [ ]:
# Elo head-to-head: pick any two teams
from atwc26_core.engines.elo import EloEngine

engine_elo = EloEngine()
engine_elo._ratings = elo_ratings

def elo_h2h(team_a: str, team_b: str):
    r = engine_elo.predict({"team_name": team_a, "home": True},
                            {"team_name": team_b, "home": False})
    ta, tb = r["team_a"], r["team_b"]
    print(f"{'─'*50}")
    print(f"  {team_a} (Elo {ta['elo_rating']:.0f})  vs  {team_b} (Elo {tb['elo_rating']:.0f})")
    print(f"  P(home win) = {ta['win_probability']*100:.1f}%")
    print(f"  P(draw)     = {r['draw_prob']*100:.1f}%")
    print(f"  P(away win) = {tb['win_probability']*100:.1f}%")
    print(f"{'─'*50}")

elo_h2h("Belgium", "Spain")
elo_h2h("France",  "England")
elo_h2h("Morocco", "Norway")


## §3 Dixon-Coles model

In [ ]:
from etl.train.dixon_coles import train_dixon_coles, save_dc_params

print("Fitting Dixon-Coles (L-BFGS-B MLE on 426 matches) …")
dc_params = train_dixon_coles(match_df)
print(f"Converged    : {dc_params['converged']}")
print(f"Home advantage γ: {dc_params['home_advantage']:.3f}  "
      f"(exp(γ)={np.exp(dc_params['home_advantage']):.3f} → "
      f"{(np.exp(dc_params['home_advantage'])-1)*100:.1f}% more goals at home)")
print(f"Avg goals    : {dc_params['avg_goals']:.2f}")
print(f"ρ (tau corr) : {dc_params['rho']:.2f}")


In [ ]:
# Attack vs defence scatter
dc_df = pd.DataFrame({
    "team":    list(dc_params["attack"].keys()),
    "attack":  list(dc_params["attack"].values()),
    "defence": list(dc_params["defence"].values()),
})

fig, ax = plt.subplots(figsize=(11, 8))
ax.scatter(dc_df["attack"], dc_df["defence"],
           s=55, color="#1D9E75", alpha=0.75, edgecolors="none", zorder=3)

# Label notable teams
label_set = (set(dc_df.nlargest(6,"attack")["team"]) |
             set(dc_df.nsmallest(6,"defence")["team"]))
for _, r in dc_df[dc_df["team"].isin(label_set)].iterrows():
    ax.annotate(r["team"], (r["attack"], r["defence"]),
                fontsize=8, xytext=(4, 2), textcoords="offset points")

ax.axvline(dc_df["attack"].mean(),  color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax.axhline(dc_df["defence"].mean(), color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xlabel("Attack α (higher = stronger attack)")
ax.set_ylabel("Defence β (lower = harder to score against)")
ax.set_title("Dixon-Coles: attack vs defence strength")

for label, x, y in [("Elite attack\n& defence",  dc_df["attack"].max()-0.4, dc_df["defence"].min()+0.03),
                     ("Weak attack\n& defence",   dc_df["attack"].min()+0.05, dc_df["defence"].max()-0.05)]:
    ax.text(x, y, label, fontsize=8, color="#888780")
plt.tight_layout()
plt.savefig("dc_full_scatter.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# Dixon-Coles scoreline matrix for one fixture
from atwc26_core.engines.dixon_coles import DixonColesEngine
import math

engine_dc = DixonColesEngine()
engine_dc._attack  = dc_params["attack"]
engine_dc._defence = dc_params["defence"]
engine_dc._home    = dc_params["home_advantage"]
engine_dc._rho     = dc_params["rho"]

def dc_scoreline_heatmap(team_a, team_b, max_g=6):
    from atwc26_core.engines.dixon_coles import _poisson_pmf, _dc_tau
    lam_h, lam_a = engine_dc._lambdas(team_a, team_b, home=True)
    matrix = np.zeros((max_g+1, max_g+1))
    for h in range(max_g+1):
        for a in range(max_g+1):
            p = _poisson_pmf(h, lam_h) * _poisson_pmf(a, lam_a)
            p *= _dc_tau(h, a, lam_h, lam_a, engine_dc._rho)
            matrix[h, a] = max(p, 0.0)
    matrix /= matrix.sum()

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(matrix * 100, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=range(max_g+1), yticklabels=range(max_g+1), ax=ax,
                cbar_kws={"label": "Probability (%)"})
    ax.set_xlabel(f"{team_b} goals")
    ax.set_ylabel(f"{team_a} goals")
    ax.set_title(f"Dixon-Coles scoreline matrix\n{team_a} (home) vs {team_b}")
    plt.tight_layout()
    plt.savefig(f"dc_matrix_{team_a[:3]}_{team_b[:3]}.png", dpi=120, bbox_inches="tight")
    plt.show()
    r = engine_dc.predict({"team_name": team_a, "home": True},
                           {"team_name": team_b, "home": False})
    print(f"P(home) {r['win_probability_a']*100:.1f}%  "
          f"P(draw) {r['draw_prob']*100:.1f}%  "
          f"P(away) {r['win_probability_b']*100:.1f}%")
    print(f"λ home={lam_h:.2f}, λ away={lam_a:.2f}")

dc_scoreline_heatmap("Belgium", "Spain")


## §4 XGBoost model

In [ ]:
from etl.train.xgboost_model import (
    build_xgb_features, train_xgboost, FEATURE_COLS, XGB_PARAMS, NUM_ROUNDS
)
import xgboost as xgb
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

X, y = build_xgb_features(match_df, elo_ratings, dc_params)
print(f"Feature matrix : {X.shape}")
print(f"Outcome dist   : {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"  0=away win, 1=draw, 2=home win")
print(f"Features: {FEATURE_COLS}")


In [ ]:
# Train and show feature importances
booster = train_xgboost(X, y)

scores = booster.get_score(importance_type="gain")
fi = (pd.DataFrame({"feature": list(scores.keys()), "gain": list(scores.values())})
        .sort_values("gain", ascending=True))

LABELS = {
    "xg_diff":          "xG differential",
    "shots_diff":       "Shots differential",
    "sot_diff":         "Shots on target diff",
    "elo_diff":         "Elo rating gap",
    "dc_attack_ratio":  "DC attack ratio (home/away)",
    "dc_defence_ratio": "DC defence ratio (away/home)",
    "home_adv":         "Home advantage",
    "h_form3":          "Home form (wins in last 3)",
    "a_form3":          "Away form (wins in last 3)",
}
fi["label"] = fi["feature"].map(LABELS).fillna(fi["feature"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fi["label"], fi["gain"], color="#EF9F27", edgecolor="none")
ax.set_xlabel("Importance (gain)")
ax.set_title(f"XGBoost feature importances  (trained on {len(y)} matches)")
plt.tight_layout()
plt.savefig("xgb_importances.png", dpi=130, bbox_inches="tight")
plt.show()
print(fi[["label","gain"]].sort_values("gain",ascending=False).to_string(index=False))


In [ ]:
# Leave-one-out cross-validation (correct approach for small dataset)
# We use 5-fold stratified CV as practical approximation of LOOCV

from sklearn.model_selection import StratifiedKFold

skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accs = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    b = xgb.train(XGB_PARAMS,
                  xgb.DMatrix(X_tr, label=y_tr, feature_names=FEATURE_COLS),
                  num_boost_round=NUM_ROUNDS, verbose_eval=False)
    preds = np.argmax(b.predict(xgb.DMatrix(X_va, feature_names=FEATURE_COLS))
                       .reshape(-1, 3), axis=1)
    acc = accuracy_score(y_va, preds)
    accs.append(acc)
    print(f"  Fold {fold+1}: {acc:.3f}")

print(f"\nCV Accuracy: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
print("(Baseline guess-home-every-time: "
      f"{(y==2).sum()/len(y):.3f}  — model should beat this)")


## §5 Poisson model — player ratings deep dive

In [ ]:
from atwc26_core.data import get_store
from atwc26_core.prediction import get_predictor, ATTACK_WEIGHTS, DEFENSE_WEIGHTS, ROLE_WEIGHTS
from atwc26_core.tournament import team_ratings, auto_pick_xi

store     = get_store()
store.load()
predictor = get_predictor(store)

# Show what the Poisson model sees per team
ratings_dict = team_ratings(store, predictor)
rat_df = pd.DataFrame([
    {"team": t, "attack": r.attack, "defense": r.defense, "gk": r.gk}
    for t, r in ratings_dict.items()
]).sort_values("attack", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))
alive = [t for t, p in
         json.loads((DATA/"winner_probabilities.json").read_text())["probabilities"].items()
         if p > 0]
rat_alive = rat_df[rat_df["team"].isin(alive)].sort_values("attack", ascending=False)

ax.scatter(rat_alive["attack"], rat_alive["defense"],
           s=80, color="#534AB7", alpha=0.8, edgecolors="none", zorder=3)
for _, r in rat_alive.iterrows():
    ax.annotate(r["team"], (r["attack"], r["defense"]),
                fontsize=8, xytext=(4, 2), textcoords="offset points")

ax.set_xlabel("Attack rating (normalised, 1.0 = tournament avg)")
ax.set_ylabel("Defence rating (normalised, 1.0 = tournament avg)")
ax.set_title("Poisson model: remaining teams — attack vs defence ratings")
ax.axvline(1.0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax.axhline(1.0, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("poisson_scatter.png", dpi=130, bbox_inches="tight")
plt.show()


## §6 All-model comparison on remaining fixtures

In [ ]:
from atwc26_core.engines import load_engines, available_engines
from atwc26_core.engines.elo import EloEngine
from atwc26_core.engines.dixon_coles import DixonColesEngine

# Inject freshly-trained params so comparison is self-consistent
load_engines(store)
engines = available_engines()

# Override Elo + DC with the ones we just trained
engine_elo_fresh = EloEngine()
engine_elo_fresh._ratings = elo_ratings
from atwc26_core.engines import register
register(engine_elo_fresh)

engine_dc_fresh = DixonColesEngine()
engine_dc_fresh._attack   = dc_params["attack"]
engine_dc_fresh._defence  = dc_params["defence"]
engine_dc_fresh._home     = dc_params["home_advantage"]
engine_dc_fresh._rho      = dc_params["rho"]
register(engine_dc_fresh)

engines = available_engines()
print(f"Models: {list(engines.keys())}")


In [ ]:
# Build comparison table for all upcoming QF/SF/Final fixtures
bracket = json.loads((DATA / "bracket.json").read_text())
upcoming = []
for rnd in bracket.get("rounds", []):
    for m in rnd["matches"]:
        if not m.get("completed"):
            a = m["slot_a"].get("team_name")
            b = m["slot_b"].get("team_name")
            if a and b:
                upcoming.append({"round": rnd["name"], "home": a, "away": b})

rows = []
for fix in upcoming:
    home, away = fix["home"], fix["away"]
    xi_h = auto_pick_xi(store.players, home)
    xi_a = auto_pick_xi(store.players, away)
    team_a = {"team_name": home, "home": True,  "players": xi_h}
    team_b = {"team_name": away, "home": False, "players": xi_a}
    for mname, engine in engines.items():
        try:
            r = engine.predict(team_a, team_b)
            rows.append({
                "round":   fix["round"],
                "home":    home,
                "away":    away,
                "model":   mname,
                "p_home":  round(r["win_probability_a"]*100, 1),
                "p_draw":  round(r["draw_probability"]*100, 1),
                "p_away":  round(r["win_probability_b"]*100, 1),
            })
        except Exception as e:
            pass

cmp_df = pd.DataFrame(rows)

# Plot comparison for each fixture
fixtures = cmp_df[["home","away"]].drop_duplicates().values
fig, axes = plt.subplots(len(fixtures), 1,
                          figsize=(11, 4 * max(len(fixtures), 1)),
                          squeeze=False)

model_colors = {"poisson":"#534AB7","elo":"#3B8BD4",
                "dixon_coles":"#1D9E75","xgboost":"#EF9F27"}

for ax, (home, away) in zip(axes[:, 0], fixtures):
    sub = cmp_df[(cmp_df["home"]==home) & (cmp_df["away"]==away)]
    x   = np.arange(len(sub))
    bar_w = 0.25
    for j, (col, label) in enumerate([("p_home","P(home win)"),
                                       ("p_draw","P(draw)"),
                                       ("p_away","P(away win)")]):
        ax.bar(x + (j-1)*bar_w, sub[col], bar_w,
               color=["#1D9E75","#EF9F27","#E24B4A"][j],
               label=label, alpha=0.85, edgecolor="none")
    ax.set_xticks(x)
    ax.set_xticklabels(sub["model"], fontsize=10)
    ax.set_ylabel("Probability (%)")
    ax.set_title(f"{home} (home) vs {away}")
    ax.legend(fontsize=9)
    ax.set_ylim(0, 90)

plt.tight_layout()
plt.savefig("model_comparison_fixtures.png", dpi=130, bbox_inches="tight")
plt.show()
display(cmp_df.pivot_table(index=["round","home","away"], columns="model",
                            values=["p_home","p_draw","p_away"], aggfunc="first"))


## §7 Winner probability simulation

In [ ]:
from atwc26_core.tournament import run_simulation

# Current output (precomputed)
wp = json.loads((DATA / "winner_probabilities.json").read_text())
probs = {t: p for t, p in wp["probabilities"].items() if p > 0}

# Re-run with Poisson only (existing)
print("Running simulation (Poisson, 1000 trials) …")
r_poisson = run_simulation(store, predictor, trials=1000, seed=42)
r_poisson_alive = {t: p for t, p in r_poisson.items() if p > 0}

print("\nPoisson simulation vs precomputed (10k trials):")
for t in sorted(probs, key=lambda x: -probs[x]):
    diff = r_poisson.get(t, 0) - probs[t]
    print(f"  {t:<20}  10k={probs[t]*100:.1f}%   1k={r_poisson.get(t,0)*100:.1f}%  "
          f"  Δ={diff*100:+.1f}pp")


In [ ]:
# Compare variance: two independent 1000-trial runs
r1 = run_simulation(store, predictor, trials=1000, seed=42)
r2 = run_simulation(store, predictor, trials=1000, seed=99)

alive_teams = [t for t in r1 if r1[t] > 0 or r2.get(t,0) > 0]
variance_df = pd.DataFrame({
    "team":  alive_teams,
    "run1":  [r1.get(t,0)*100 for t in alive_teams],
    "run2":  [r2.get(t,0)*100 for t in alive_teams],
})
variance_df["|Δ|pp"] = abs(variance_df["run1"] - variance_df["run2"]).round(2)
variance_df = variance_df.sort_values("run1", ascending=False).reset_index(drop=True)

print("Variance between two 1000-trial runs (±pp):")
display(variance_df)
print(f"\nMax variance: {variance_df['|Δ|pp'].max():.1f}pp  "
      f"(mean: {variance_df['|Δ|pp'].mean():.1f}pp)")
print("At 10,000 trials this variance is ~3x smaller.")


## §8 Calibration analysis (post-tournament)

> **Run this section after the Final** — it compares predicted
> probabilities against actual match outcomes to measure model accuracy.


In [ ]:
# Calibration: for each completed bracket match, did the higher-probability team win?
bracket   = json.loads((DATA / "bracket.json").read_text())
bp        = json.loads((DATA / "bracket_predictions.json").read_text())
completed = []

for rnd in bracket.get("rounds", []):
    for m in rnd["matches"]:
        if not m.get("completed"):
            continue
        gid  = str(m.get("game_id",""))
        pred = bp.get("predictions", {}).get(gid)
        if not pred or pred.get("win_probability") is None:
            continue
        # bracket_predictions uses team_a_name/team_b_name (not slot_a/slot_b)
        pred_winner  = pred.get("predicted_winner")
        real_score_a = int(m.get("score_a") or 0)
        real_score_b = int(m.get("score_b") or 0)
        if real_score_a == real_score_b:
            continue
        # slot_a is the "home" side of the match
        real_winner = m["slot_a"]["team_name"] if real_score_a > real_score_b \
                      else m["slot_b"]["team_name"]
        completed.append({
            "round":       rnd["name"],
            "home":        pred.get("team_a_name"),
            "away":        pred.get("team_b_name"),
            "pred_winner": pred_winner,
            "real_winner": real_winner,
            "p_winner":    pred.get("win_probability"),
            "correct":     pred_winner == real_winner,
        })

if not completed:
    print("No completed bracket matches with predictions yet.")
    print("Re-run this cell after the Final to see calibration results.")
else:
    cal_df = pd.DataFrame(completed)
    acc    = cal_df["correct"].mean()
    print(f"Bracket matches predicted : {len(cal_df)}")
    print(f"Accuracy (favourite won)  : {acc*100:.1f}%")
    print(f"Avg P(predicted winner)   : {cal_df['p_winner'].mean()*100:.1f}%")
    display(cal_df[["round","home","away","pred_winner","real_winner","p_winner","correct"]])
